In [24]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

github_url = "https://raw.githubusercontent.com/ayaanjauhar/excel_python_project_2/main/Excel_Handson2_Data_07.10.26.xlsx"
local_file = "Excel_Handson2_Data_07.10.26.xlsx"

if os.path.exists(local_file):
    file = local_file
else:
    file = github_url

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

print(f"Dataset successfully connected: {file}")

Dataset successfully connected: Excel_Handson2_Data_07.10.26.xlsx


In [25]:
df = pd.read_excel(file, sheet_name="P1 Salaries")
sal = df["Monthly salary (Rs 000)"]
print("Mean :", sal.mean())
print("Median:", sal.median())
print("Mode :", sal.mode().tolist())
# Budget for 30 new hires for one year (12 months)
print("Budget using mean :", sal.mean() * 30 * 12)
print("Budget using median:", sal.median() * 30 * 12)

Mean : 74.4
Median: 60.0
Mode : [45]
Budget using mean : 26784.0
Budget using median: 21600.0


In [26]:
df = pd.read_excel(file, sheet_name="P2 Weighted")
courses = df[["Course", "Credits", "Grade points"]]
print("Simple mean of grade points:", courses["Grade points"].mean())
print("CGPA (weighted by credits) :",
np.average(courses["Grade points"], weights=courses["Credits"]))
rice = df[["Quantity (kg)", "Price (Rs per kg)"]].dropna()
print("Simple mean price :", rice["Price (Rs per kg)"].mean())
print("Weighted mean price:",
np.average(rice["Price (Rs per kg)"], weights=rice["Quantity (kg)"]))

Simple mean of grade points: 7.666666666666667
CGPA (weighted by credits) : nan
Simple mean price : 47.25
Weighted mean price: 44.325


In [27]:
df = pd.read_excel(file, sheet_name="P3 Scores")
score = df["Score (out of 100)"]

for p in [10, 25, 50, 75, 80, 85, 90, 95]:
    textbook = np.percentile(score, p, method="averaged_inverted_cdf")
    excel_inc = np.percentile(score, p)  # same as PERCENTILE.INC
    print(p, "textbook:", textbook, "Excel INC:", round(excel_inc, 2))

q1, q3 = np.percentile(score, [25, 75], method="averaged_inverted_cdf")
print("IQR:", q3 - q1)
print("Share of students scoring below 75:", (score < 75).mean())

10 textbook: 50.0 Excel INC: 50.0
25 textbook: 55.0 Excel INC: 55.0
50 textbook: 64.0 Excel INC: 64.0
75 textbook: 71.0 Excel INC: 71.0
80 textbook: 72.5 Excel INC: 72.2
85 textbook: 74.0 Excel INC: 74.0
90 textbook: 75.5 Excel INC: 75.1
95 textbook: 78.5 Excel INC: 78.05
IQR: 16.0
Share of students scoring below 75: 0.8833333333333333


In [28]:
df4 = pd.read_excel(file, sheet_name="P4 Variability")


def spread(x):
    x = x.dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    return pd.Series({
        "mean": x.mean(),
        "range": x.max() - x.min(),
        "IQR": q3 - q1,
        "var (sample)": x.var(),
        "sd (sample)": x.std(),
        "var (population)": x.var(ddof=0),
        "sd (population)": x.std(ddof=0),
        "CV %": x.std() / x.mean() * 100,
    })


cols = [
    "Machine A (ml)",
    "Machine B (ml)",
    "Product X (units)",
    "Product Y (units)",
]
print(df4[cols].apply(spread).round(3))

                  Machine A (ml)  Machine B (ml)  Product X (units)  Product Y (units)
mean                     499.600         499.750            463.857             17.143
range                      5.100           9.600             94.000             15.000
IQR                        2.500           5.150             59.500              6.250
var (sample)               2.962          10.486           1028.286             18.132
sd (sample)                1.721           3.238             32.067              4.258
var (population)           2.715           9.613            954.837             16.837
sd (population)            1.648           3.100             30.900              4.103
CV %                       0.344           0.648              6.913             24.839


In [29]:
df5 = pd.read_excel(file, sheet_name="P5 Shape")
ret = df5["Daily return (%)"].dropna()
ht = df5["Height (cm)"].dropna()
sal = pd.read_excel(file, sheet_name="P1 Salaries")["Monthly salary (Rs 000)"]

data = [("Salaries", sal), ("Returns", ret), ("Heights", ht)]

for name, x in data:
    print(
        f"{name:9s} mean={x.mean():7.2f} median={x.median():7.2f}"
        f" skew={x.skew():6.2f} excess kurt={x.kurt():6.2f}"
    )

Salaries  mean=  74.40 median=  60.00 skew=  2.75 excess kurt=  8.16
Returns   mean=  -0.06 median=   0.01 skew= -1.86 excess kurt=  7.16
Heights   mean= 165.33 median= 165.00 skew=  0.21 excess kurt= -0.06


In [30]:
df6 = pd.read_excel(file, sheet_name="P6 Sales")
sales = df6["Monthly sales (Rs 000)"]

# Compute z-scores
df6["z"] = (sales - sales.mean()) / sales.std()
df6["abs_z"] = df6["z"].abs()

print(df6[["Sales rep", "Monthly sales (Rs 000)", "z", "abs_z"]])

# Filter out outliers (|z| > 3)
df6_clean = df6[df6["abs_z"] <= 3]
print("\n--- After Removing Outlier ---")
print("Clean Mean:", round(df6_clean["Monthly sales (Rs 000)"].mean(), 2))
print("Clean SD:", round(df6_clean["Monthly sales (Rs 000)"].std(), 2))

   Sales rep  Monthly sales (Rs 000)         z     abs_z
0      Kunal                      82  0.218535  0.218535
1      Pooja                      72 -0.450449  0.450449
2      Anaya                      68 -0.718043  0.718043
3     Aditya                      58 -1.387028  1.387028
4     Ishaan                      82  0.218535  0.218535
5       Isha                      77 -0.115957  0.115957
6       Diya                      85  0.419230  0.419230
7         Om                      90  0.753722  0.753722
8      Arjun                      63 -1.052535  1.052535
9    Reyansh                      76 -0.182856  0.182856
10      Myra                      68 -0.718043  0.718043
11     Krish                      89  0.686824  0.686824
12    Simran                      73 -0.383551  0.383551
13      Avni                      86  0.486129  0.486129
14    Vikram                     142  4.232441  4.232441
15     Meera                      80  0.084738  0.084738
16     Aditi                   

In [ ]:
import numpy as np
import pandas as pd

# 1. Read sheet P7 Rents
df7 = pd.read_excel(file, sheet_name="P7 Rents")

# 2. Select rent column using your exact column header
rent = df7["Monthly rent (Rs 000)"].dropna()

# 3. Calculate quartiles and IQR
q1, q3 = rent.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

# 4. Filter outliers
outliers = rent[(rent < lower_fence) | (rent > upper_fence)]

print("Q1:", round(q1, 2), "Q3:", round(q3, 2), "IQR:", round(iqr, 2))
print("Lower Fence:", round(lower_fence, 2))
print("Upper Fence:", round(upper_fence, 2))
print("Outliers count:", len(outliers))
print("Outlier values:", outliers.tolist())

Q1: 29.5 Q3: 36.0 IQR: 6.5
Lower Fence: 19.75
Upper Fence: 45.75
Outliers count: 2
Outlier values: [78, 95]


In [1]:
import os
import pandas as pd

# Define GitHub raw URL & local filename
github_url = "https://raw.githubusercontent.com/ayaanjauhar/excel_python_project_2/main/Excel_Handson2_Data_07.10.26.xlsx"
local_file = "Excel_Handson2_Data_07.10.26.xlsx"

# Use local file if available (VS Code), otherwise fetch from GitHub (Google Colab)
file = local_file if os.path.exists(local_file) else github_url

# 1. Read sheet P8 Students
df8 = pd.read_excel(file, sheet_name="P8 Students")

# Strip extra spaces from column names
df8.columns = df8.columns.str.strip()

# 2. Select columns matching your exact headers
study_h = df8["Study hours per week"]
social_h = df8["Social media hours per day"]
marks = df8["Exam marks"]
study_m = df8["Study Minutes"]

# 3. Calculate Covariances and Correlations
print("--- Problem 8 Results ---")
print(
    "Covariance (Study hours, Exam marks):",
    round(study_h.cov(marks), 2),
)
print(
    "Correlation (Study hours, Exam marks):",
    round(study_h.corr(marks), 2),
)

print(
    "Covariance (Social media hours, Exam marks):",
    round(social_h.cov(marks), 2),
)
print(
    "Correlation (Social media hours, Exam marks):",
    round(social_h.corr(marks), 2),
)

print(
    "Correlation (Study hours, Social media hours):",
    round(study_h.corr(social_h), 2),
)

print(
    "\nCovariance (Study minutes, Exam marks):",
    round(study_m.cov(marks), 2),
)
print(
    "Correlation (Study minutes, Exam marks):",
    round(study_m.corr(marks), 2),
)

--- Problem 8 Results ---
Covariance (Study hours, Exam marks): 57.1
Correlation (Study hours, Exam marks): 0.89
Covariance (Social media hours, Exam marks): -9.43
Correlation (Social media hours, Exam marks): -0.58
Correlation (Study hours, Social media hours): -0.8

Covariance (Study minutes, Exam marks): 3425.71
Correlation (Study minutes, Exam marks): 0.89
